# Baic analysis - Xenium Ovarian Cancer

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import contextlib
import io
import os
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

import spatialdata as sd
from spatialdata_io import xenium
import scanpy as sc
import squidpy as sq

warnings.filterwarnings("ignore")

xenium_path = "./Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs"
zarr_path   = "./Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs.zarr"
h5ad_path   = "./adata_processed.h5ad"

start_time = time.time()
last_time  = [start_time]

def step(name):
    now = time.time()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {name} | "
          f"step: {now - last_time[0]:.2f}s | total: {now - start_time:.2f}s")
    last_time[0] = now


def spatial_clusters(adata, key, *, size=1.5, alpha=0.8, palette=None):
    n = adata.obs[key].nunique()
    if isinstance(palette, dict):
        cats = (adata.obs[key].cat.categories if hasattr(adata.obs[key], "cat")
                else sorted(adata.obs[key].dropna().unique()))
        palette = ListedColormap([palette.get(c, "#7f7f7f") for c in cats])
    elif palette is None:
        palette = "tab20" if n > 10 else "tab10"
    print(f"[spatial] {key}: {n} clusters -> "
          f"{adata.obs[key].value_counts().sort_index().to_dict()}")
    with contextlib.redirect_stdout(io.StringIO()):
        return sq.pl.spatial_scatter(
            adata, shape=None, color=key, size=size, alpha=alpha,
            palette=palette, frameon=False, wspace=0.4,
        )

step("setup done")

In [ ]:
if not os.path.exists(zarr_path):
    sdata = xenium(xenium_path)
    step("xenium loaded")
    sdata.write(zarr_path, overwrite=True)
    step("zarr written")
else:
    step("zarr already exists, skipping xenium conversion")

In [ ]:
sdata = sd.read_zarr(zarr_path)
step("zarr read")

In [ ]:
adata = sdata.tables["table"]

sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
step("qc metrics computed")

cprobes = adata.obs["control_probe_counts"].sum()  / adata.obs["total_counts"].sum() * 100
cwords  = adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")
step("control probes/codewords printed")

> Background is negligible: negative controls account for <1% of counts.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])

axs[2].set_title("Area of segmented cells")
sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(adata.obs["nucleus_area"] / adata.obs["cell_area"], kde=False, ax=axs[3])

for ax in axs:
    ax.text(
        0.01, 0.95,
        f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
    )

plt.tight_layout()
step("qc plots rendered")

In [ ]:
print(f"Before filtering: {adata.n_obs} cells x {adata.n_vars} genes")
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
print(f"After filtering:  {adata.n_obs} cells x {adata.n_vars} genes")
step("cells and genes filtered")

> Filtering drops ~4% of cells; all 480 genes survive.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
step("normalized and log1p")

In [ ]:
sc.pp.pca(adata)
step("pca done")

sc.pp.neighbors(adata)
step("neighbors done")

sc.tl.umap(adata)
step("umap done")

sc.tl.leiden(
    adata,
    resolution=1.0,
    key_added="leiden",
    flavor="leidenalg",
    n_iterations=2,
)
print(f"Leiden found {adata.obs['leiden'].nunique()} clusters")
step("leiden done")

> Resolution 1.0 yields six clusters.

In [ ]:
adata.write(h5ad_path)
step("adata checkpoint saved")

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "leiden"],
    wspace=0.4,
    show=False,
)
step("umap plots rendered")

In [ ]:
spatial_clusters(adata, "leiden")

with contextlib.redirect_stdout(io.StringIO()):
    sq.pl.spatial_scatter(
        adata, shape=None, color="total_counts",
        size=1.5, alpha=0.8, frameon=False,
    )
step("spatial plots rendered")

In [ ]:
from loguru import logger
logger.remove()

from napari_spatialdata import Interactive

interactive = Interactive(sdata)
interactive.add_element("morphology_focus", "global")
interactive.add_element("cell_labels", "global")
interactive.run()

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    method="wilcoxon",
    key_added="rank_leiden",
    pts=True,
)
step("rank_genes_groups (wilcoxon) done")

rg = sc.get.rank_genes_groups_df(adata, group=None, key="rank_leiden")
print(f"rank_genes_groups rows: {len(rg)} (groups x genes)")
print(rg.head())
print(f"leiden groups: {sorted(adata.obs['leiden'].unique(), key=int)}")
step("rank results summarized")

In [ ]:
n_top    = 3
min_lfc  = 0.25
max_padj = 0.05

sig = rg[(rg["pvals_adj"] < max_padj) & (rg["logfoldchanges"] > min_lfc)].copy()
print(f"Significant genes: {len(sig)} / {len(rg)}")

top = (
    sig.sort_values("scores", ascending=False)
       .groupby("group", observed=True)
       .head(n_top)
)

marker_genes = top["names"].drop_duplicates().tolist()
print(f"Selected {len(marker_genes)} unique marker genes across {top['group'].nunique()} clusters")
print("Markers per cluster:")
print(top.groupby("group", observed=True)["names"].apply(list).to_string())
step("marker_genes selected")

In [ ]:
dp = sc.pl.dotplot(
    adata,
    var_names=marker_genes,
    groupby="leiden",
    standard_scale="var",
    cmap="Reds",
    dendrogram=True,
    figsize=(max(6, 0.35 * len(marker_genes)), 8),
    return_fig=True,
    show=False,
)
dp.show()
step("dotplot rendered")

In [ ]:
leiden   = adata.obs["leiden"].astype(str)
clusters = sorted(leiden.unique(), key=lambda x: int(x) if x.isdigit() else x)

X = adata.X.tocsr() if sp.issparse(adata.X) else np.asarray(adata.X)
gene_idx = adata.var_names.get_indexer(marker_genes)
assert (gene_idx >= 0).all(), f"missing genes: {[g for g, i in zip(marker_genes, gene_idx) if i < 0]}"

mean = np.zeros((len(clusters), len(marker_genes)))
frac = np.zeros_like(mean)
for i, c in enumerate(clusters):
    m = (leiden == c).values
    sub = X[m][:, gene_idx]
    mean[i] = np.asarray(sub.mean(axis=0)).ravel()
    frac[i] = np.asarray((sub > 0).sum(axis=0)).ravel() / m.sum()

frac_df = pd.DataFrame(frac, index=clusters, columns=marker_genes)
mean_df = pd.DataFrame(mean, index=clusters, columns=marker_genes)

print(f"cluster sizes: {leiden.value_counts().sort_index().to_dict()}")
print(f"frac_df  shape={frac_df.shape}  range=[{frac.min():.3f}, {frac.max():.3f}]")
print(f"mean_df  shape={mean_df.shape}  range=[{mean.min():.3f}, {mean.max():.3f}]")

rg_wide = rg.pivot(index="group", columns="names", values="pct_nz_group")
rows = [r for r in frac_df.index if r in rg_wide.index]
cols = [c for c in frac_df.columns if c in rg_wide.columns]
delta = (frac_df.loc[rows, cols] - rg_wide.loc[rows, cols]).abs()
print(f"max |frac_df - pct_nz_group| = {delta.values.max():.2e}")

per_gene = pd.DataFrame({
    "max_frac": frac_df.max(axis=0),
    "argmax":   frac_df.idxmax(axis=0),
    "max_mean": mean_df.max(axis=0),
}).sort_values("max_frac", ascending=False)
print("\nPer-marker summary (top 15 by max fraction):")
print(per_gene.head(15).round(3).to_string())

per_cluster = pd.DataFrame({
    "n_markers_frac>0.5": (frac_df > 0.5).sum(axis=1),
    "n_markers_frac>0.2": (frac_df > 0.2).sum(axis=1),
    "mean_frac":          frac_df.mean(axis=1),
})
print("\nPer-cluster marker coverage:")
print(per_cluster.round(3).to_string())

print("\nFull fraction matrix (fraction of cells expressing):")
print(frac_df.round(2).to_string())
step("dotplot numbers inspected")

> Cluster 4 has no marker above 6% detection — likely debris. Cluster 2 is ACTB-dominated — a low-depth epithelial tail.

In [ ]:
profiles = np.vstack([
    np.asarray(X[(leiden == c).values].mean(axis=0)).ravel() for c in clusters
])
corr = np.corrcoef(profiles)
corr_df = pd.DataFrame(corr, index=clusters, columns=clusters)

off = corr.copy()
np.fill_diagonal(off, -np.inf)
flat = np.argsort(off.ravel())[::-1]

seen, pairs = set(), []
for k in flat:
    i, j = np.unravel_index(k, corr.shape)
    if i < j and (i, j) not in seen:
        seen.add((i, j))
        pairs.append((clusters[i], clusters[j], corr[i, j]))
    if len(pairs) == 5:
        break

print("Top 5 most similar cluster pairs (Pearson r on mean log-norm profile):")
for a, b, r in pairs:
    print(f"  cluster {a}  vs  cluster {b} :  r = {r:.4f}")

print("\nFull correlation matrix:")
print(corr_df.round(3).to_string())
step("cluster similarity computed")

> Clusters 0 and 1 are the same cell type (r = 0.988), split by sequencing depth.

In [ ]:
qc = adata.obs.groupby("leiden", observed=True).agg(
    n_cells          = ("leiden", "size"),
    total_counts_med = ("total_counts", "median"),
    total_counts_p10 = ("total_counts", lambda x: np.percentile(x, 10)),
    n_genes_med      = ("n_genes_by_counts", "median"),
    cell_area_med    = ("cell_area", "median"),
    nucleus_ratio    = ("nucleus_area", lambda x: np.median(x / adata.obs.loc[x.index, "cell_area"])),
).round(2)
print(qc.to_string())
step("per-cluster QC inspected")

In [ ]:
merge_map = {"0": "0+1", "1": "0+1"}
adata.obs["leiden_m"] = adata.obs["leiden"].astype(str).replace(merge_map)

print("merged cluster sizes:")
print(adata.obs["leiden_m"].value_counts().sort_index().to_string())

sc.pl.dotplot(
    adata, var_names=marker_genes, groupby="leiden_m",
    standard_scale="var", cmap="Reds", dendrogram=True,
    figsize=(max(6, 0.35 * len(marker_genes)), 6), show=False,
)
step("merged dotplot rendered")

In [ ]:
n_top    = 2
min_lfc  = 1.0
min_frac = 0.10

sig = rg[
    (rg["pvals_adj"] < 0.05)
    & (rg["logfoldchanges"] > min_lfc)
    & (rg["pct_nz_group"] > min_frac)
].copy()
print(f"significant & meaningful: {len(sig)} / {len(rg)}")

top = sig.sort_values("scores", ascending=False).groupby("group", observed=True).head(n_top)
marker_genes2 = top["names"].drop_duplicates().tolist()
print("tightened markers:", marker_genes2)
print(top.groupby("group", observed=True)["names"].apply(list).to_string())

sc.pl.dotplot(
    adata, var_names=marker_genes2, groupby="leiden_m",
    standard_scale="var", cmap="Reds", dendrogram=True,
    figsize=(max(6, 0.35 * len(marker_genes2)), 6), show=False,
)
step("tightened markers plotted")

In [ ]:
spatial_clusters(adata, "leiden")
spatial_clusters(adata, "leiden_m")

with contextlib.redirect_stdout(io.StringIO()):
    sq.pl.spatial_scatter(
        adata, shape=None, color="total_counts",
        size=1.5, alpha=0.8, frameon=False,
    )
step("spatial plots rendered (leiden + leiden_m + total_counts)")

In [ ]:
lab = adata.obs["leiden"].astype(str).values
colors = np.select(
    [lab == "0", lab == "1"],
    ["#1f77b4", "#ff7f0e"],
    default="#dddddd",
)
xy = np.asarray(adata.obsm["spatial"])[:, :2]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=1.2, linewidths=0)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.axis("off")
ax.set_title("cluster 0 (blue) vs cluster 1 (orange), rest grey")
plt.tight_layout()
plt.show()
step("0 vs 1 spatial overlay rendered")

In [ ]:
adata_scaled = adata.copy()

sc.pp.scale(adata_scaled, max_value=10)
sc.pp.pca(adata_scaled, n_comps=30)
sc.pp.neighbors(adata_scaled, n_neighbors=15)
sc.tl.leiden(
    adata_scaled, resolution=1.0, key_added="leiden_scaled",
    flavor="leidenalg", n_iterations=2,
)

old = adata.obs["leiden"].astype(str)
new = adata_scaled.obs["leiden_scaled"].astype(str)
mask = old.isin(["0", "1"])

ct = pd.crosstab(old[mask], new[mask])
print("Cross-tab of old (0/1) vs new clusters for those cells:")
print(ct.to_string())

max_col = ct.sum(axis=0).idxmax()
frac    = ct.loc[:, max_col].sum() / ct.values.sum()
print(f"\n{frac:.1%} of old 0+1 cells land in new cluster {max_col}")
step("scaled PCA/leiden hypothesis test done")

> Z-scaling doesn't collapse 0+1 — the split is depth, not structure.

In [ ]:
annotation_map = {
    "0+1": "Epithelial / Tumor",
    "2":   "Epithelial (Low-count)",
    "3":   "Fibroblast / CAF",
    "4":   "Low-content / Artifact",
    "5":   "Macrophage / Myeloid",
}

adata.obs["cell_type"] = adata.obs["leiden_m"].map(annotation_map).astype("category")

print("Cell type counts:")
print(adata.obs["cell_type"].value_counts().to_string())
print(f"Missing labels: {adata.obs['cell_type'].isna().sum()} cells")
step("clusters annotated")

> Final: five cell types; epithelial fraction dominates at ~67%.

In [ ]:
pal = {
    "Epithelial / Tumor":     "#d62728",
    "Epithelial (Low-count)": "#ff9896",
    "Fibroblast / CAF":       "#2ca02c",
    "Macrophage / Myeloid":   "#9467bd",
    "Low-content / Artifact": "#7f7f7f",
}

sc.pl.umap(
    adata, color="cell_type", palette=pal,
    size=5, legend_loc="on data", frameon=False, show=False,
)
plt.title("UMAP: Annotated Cell Types")
plt.show()

spatial_clusters(adata, "cell_type", palette=pal, size=1.5)
step("annotated UMAP and spatial plots rendered")

In [ ]:
signatures = {
    "Epithelial": ["EPCAM", "KRAS", "ERBB3"],
    "Stromal":    ["SPARC", "FN1", "IGFBP7"],
    "Myeloid":    ["CD74", "APOE", "CTSD"],
}

for name, genes in signatures.items():
    valid = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(adata, valid, score_name=f"{name}_score")

sc.pl.stacked_violin(
    adata,
    var_names=[f"{n}_score" for n in signatures],
    groupby="cell_type",
    cmap="viridis",
    figsize=(8, 4),
    show=False,
)
plt.title("Signature Scores per Annotated Cell Type")
plt.show()
step("gene signature scores validated")

In [ ]:
annotated_h5ad_path = "./adata_annotated.h5ad"
adata.write(annotated_h5ad_path)
print(f"Annotated data saved to {annotated_h5ad_path}")

summary = adata.obs.groupby("cell_type", observed=True).agg(
    n_cells=("cell_type", "size"),
    pct_total=("cell_type", lambda x: len(x) / len(adata) * 100),
    median_counts=("total_counts", "median"),
    median_genes=("n_genes_by_counts", "median"),
).round(2)
print("\nFinal Annotation Summary:")
print(summary.to_string())
step("annotated data exported")